# 📊 Session 2: Analysis, Correlation & Anomalies
**Data Science Workshop · Part 2 of 2 · ~60 minutes**

> **📦 The scenario continues:** The ShopSmart data is clean. Now your manager asks:
> 1. *What drives our revenue?*
> 2. *Why do some customers give bad ratings?*
> 3. *Are there suspicious orders we should look at?*
> 4. *Can we turn this into something that runs every month?*

### 🗺️ Agenda
| ⏱ | Section | You will learn |
|---|---|---|
| 0–15 | 1. Exploratory Data Analysis | Distributions, skew, groupby, pivots, time series |
| 15–30 | 2. Correlation | Pearson vs Spearman, heatmaps, pitfalls, causation |
| 30–45 | 3. Anomalies | Rules, z-score, IQR, contextual, Isolation Forest |
| 45–55 | 4. Data engineering | Pipelines, validation, bronze/silver/gold layers, feature engineering, feature scaling |
| 55–60 | 5. Wrap-up | Insights for the CEO, next steps |


| Icon | Meaning |
|---|---|
| 📖 | Concept / theory |
| 💻 | Run this code and look at the output |
| 🧪 | Exercise: your turn! |
| ⚠️ | Common pitfall |
| 💡 | Tip / best practice |
| 🚀 | Stretch goal (for advanced participants, optional) |

**How to run a cell:** click it and press **Shift + Enter**. To start fresh, use **Kernel → Restart Kernel and Run All Cells**.

## ⚙️ Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 4)

# Works whether the notebook is in the workshop root or in /solutions
DATA = Path("data") if Path("data").exists() else Path("..") / "data"
print("pandas", pd.__version__, "| data folder:", DATA.resolve())

In [ ]:
path = DATA / "orders_clean.parquet"
if not path.exists():
    raise FileNotFoundError("orders_clean.parquet not found. Run the Session 1 notebook first.")
df = pd.read_parquet(path)
print(df.shape)
df.head()

---
## 1. 🔭 Exploratory Data Analysis (EDA) *(⏱ 15 min)*

📖 EDA means **asking questions of the data** before building anything. A useful order:
1. **Univariate**: one variable at a time (distribution, center, spread)
2. **Bivariate**: two variables (relationships, group differences)
3. **Multivariate**: many variables together

### 1.1 Numerical distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(df["total_amount"], bins=60, ax=axes[0]).set_title("Order value (raw scale)")
sns.histplot(np.log10(df["total_amount"]), bins=60, ax=axes[1]).set_title("Order value (log10 scale)")
plt.show()

s = df["total_amount"]
print(f"mean = {s.mean():,.0f}   median = {s.median():,.0f}   skew = {s.skew():.1f}")

📖 **Right-skewed** data (a long tail of big orders) pulls the **mean** above the **median**.
💡 For "typical" values in skewed data, report the **median**. A **log scale** often reveals the shape better.

### 1.2 Categorical distribution

In [ ]:
ax = df["category"].value_counts().plot.bar(rot=0, color="tab:blue")
ax.set_title("Orders per category")
plt.show()

### 1.3 Numerical vs categorical: comparing groups

In [ ]:
ax = sns.boxplot(data=df, x="category", y="total_amount", log_scale=True)
ax.set_title("Order value by category (log scale)")
plt.show()

In [ ]:
summary = (
    df.groupby("category")
      .agg(orders=("order_id", "count"),
           revenue=("total_amount", "sum"),
           median_order=("total_amount", "median"),
           avg_rating=("customer_rating", "mean"),
           return_rate=("returned", "mean"))
      .sort_values("revenue", ascending=False)
)
summary.style.format({"revenue": "${:,.0f}", "median_order": "${:,.0f}",
                      "avg_rating": "{:.2f}", "return_rate": "{:.1%}"}).background_gradient(subset=["revenue"])

### 1.4 Pivot table: two dimensions at once

In [ ]:
pivot = df.pivot_table(index="city", columns="category", values="total_amount", aggfunc="sum")
sns.heatmap(pivot / 1000, annot=True, fmt=".0f", cmap="Blues", cbar_kws={"label": "Revenue ($K)"})
plt.title("Revenue ($K) by city × category")
plt.show()

### 1.5 Time series: is there seasonality?

In [ ]:
monthly = df.set_index("order_date").resample("ME").agg({"total_amount": "sum", "discount": "mean"})

fig, ax1 = plt.subplots(figsize=(12, 4))
ax1.bar(monthly.index.strftime("%b"), monthly["total_amount"] / 1000, color="tab:blue", alpha=0.7)
ax1.set_ylabel("Revenue ($K)")
ax2 = ax1.twinx()
ax2.plot(monthly.index.strftime("%b"), monthly["discount"] * 100, color="tab:red", marker="o")
ax2.set_ylabel("Avg discount (%)", color="tab:red")
ax2.grid(False)
plt.title("Monthly revenue vs average discount")
plt.show()

💡 Revenue spikes in **Nov–Dec**, exactly when discounts go up. Is that cause and effect? Keep that question for Section 2!

### 🧪 Exercise 1.1: Shipping

For each `shipping_method`, compute the **median delivery days**, **average rating** and **return rate**. What do you notice?

In [ ]:
# your code here

### 🧪 Exercise 1.2 🚀: Weekday pattern

Which **weekday** has the most orders? (hint: `df["order_date"].dt.day_name()`)

In [ ]:
# your code here

---
## 2. 🔗 Correlation *(⏱ 15 min)*

📖 A **correlation coefficient** *r* goes from **−1** to **+1**:

| \|r\| | Strength |
|---|---|
| 0.0 – 0.1 | none |
| 0.1 – 0.3 | weak |
| 0.3 – 0.5 | moderate |
| > 0.5 | strong |

- **Sign** = direction (both go up together, or one goes up while the other goes down)
- **Pearson**: measures **linear** relationships and is **sensitive to outliers**
- **Spearman**: works on **ranks**, so it captures any **monotonic** relationship and is **robust to outliers**

In [ ]:
num_cols = ["quantity", "unit_price", "discount", "shipping_cost", "delivery_days",
            "session_minutes", "customer_age", "customer_rating", "total_amount"]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, method in zip(axes, ["pearson", "spearman"]):
    corr = df[num_cols].corr(method=method)
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", center=0,
                vmin=-1, vmax=1, ax=ax, cbar=False)
    ax.set_title(method.capitalize())
plt.tight_layout()
plt.show()

🤔 **Discuss:** Why are some Pearson and Spearman values so different?
(Hint: a few extreme values, which we'll find in Section 3, distort Pearson.)

### 2.1 Delivery time vs rating

In [ ]:
d = df[df["delivery_days"] <= 20]          # zoom in on the normal range
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.regplot(data=d, x="delivery_days", y="customer_rating", x_jitter=0.3, y_jitter=0.2,
            scatter_kws={"alpha": 0.15, "s": 10}, line_kws={"color": "red"}, ax=axes[0])
d.groupby("delivery_days")["customer_rating"].mean().plot(marker="o", ax=axes[1])
axes[1].set_ylabel("avg rating")
plt.show()

print("Pearson :", round(df["delivery_days"].corr(df["customer_rating"]), 2))
print("Spearman:", round(df["delivery_days"].corr(df["customer_rating"], method="spearman"), 2))

### 2.2 Pearson vs Spearman on a non-linear relationship

In [ ]:
d = df[df["session_minutes"] < 120]
ax = sns.scatterplot(data=d, x="total_amount", y="session_minutes", alpha=0.2, s=10)
ax.set_xscale("log")
plt.show()

for name, data in [("all rows", df), ("without extreme sessions", d)]:
    p = data["session_minutes"].corr(data["total_amount"])
    sp = data["session_minutes"].corr(data["total_amount"], method="spearman")
    print(f"{name:<26} Pearson={p:.2f}  Spearman={sp:.2f}")

### ⚠️ 2.3 Three correlation pitfalls
**Pitfall 1: r ≈ 0 does NOT mean "no relationship"**

In [ ]:
rng = np.random.default_rng(0)
x = rng.uniform(-3, 3, 300)
y = x ** 2 + rng.normal(0, 0.5, 300)
plt.figure(figsize=(5, 3.5))
plt.scatter(x, y, s=8)
plt.title(f"Pearson r = {np.corrcoef(x, y)[0, 1]:.2f}, yet y depends on x perfectly!")
plt.show()

**Pitfall 2: one outlier can create a correlation**

In [ ]:
x = rng.normal(0, 1, 30)
y = rng.normal(0, 1, 30)
x_out, y_out = np.append(x, 15), np.append(y, 15)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].scatter(x, y); axes[0].set_title(f"30 random points: r = {np.corrcoef(x, y)[0, 1]:.2f}")
axes[1].scatter(x_out, y_out); axes[1].set_title(f"+1 outlier: r = {np.corrcoef(x_out, y_out)[0, 1]:.2f}")
plt.show()

**Pitfall 3: correlation ≠ causation** (confounders)

In [ ]:
temperature = rng.uniform(10, 40, 365)
ice_cream = 20 * temperature + rng.normal(0, 60, 365)
sunburns = 0.5 * temperature + rng.normal(0, 2, 365)
print(f"corr(ice cream, sunburns) = {np.corrcoef(ice_cream, sunburns)[0, 1]:.2f}  🍦☀️")

# Control for the confounder: remove the effect of temperature from both, then correlate what's left
res_ice = ice_cream - np.polyval(np.polyfit(temperature, ice_cream, 1), temperature)
res_sun = sunburns - np.polyval(np.polyfit(temperature, sunburns, 1), temperature)
print(f"after controlling for temperature = {np.corrcoef(res_ice, res_sun)[0, 1]:.2f}")

💡 Ice cream doesn't cause sunburns; **temperature** drives both. In our data: do discounts *cause* the Nov–Dec revenue spike, or is it the **holiday season** driving both?
Only an **experiment** (A/B test) can prove causation.

### 2.4 Categorical ↔ numerical association

In [ ]:
ax = df.groupby("customer_rating")["returned"].mean().mul(100).plot.bar(rot=0, color="tab:orange")
ax.set_ylabel("Return rate (%)")
ax.set_title("Return rate by rating")
plt.show()

### 🧪 Exercise 2.1: Strongest driver of ratings

Which numeric column has the strongest **absolute** Spearman correlation with `customer_rating`?

In [ ]:
# hint: df[num_cols].corr(method="spearman")["customer_rating"]

### 🧪 Exercise 2.2 🚀: Discount and quantity

Do bigger discounts go with bigger quantities? Compare Pearson and Spearman for `discount` vs `quantity`, and plot the median quantity per discount level.

In [ ]:
# your code here

---
## 3. 🚨 Anomaly detection *(⏱ 15 min)*

📖 An **anomaly** (outlier) is an observation that differs a lot from the rest.
| Type | Example |
|---|---|
| **Point** | One order with quantity = 200 |
| **Contextual** | $500 is normal for a laptop but crazy for a book |
| **Collective** | 50 orders from one card in 5 minutes |

| Approach | Methods |
|---|---|
| Rules (domain knowledge) | "delivery > 30 days is impossible" |
| Statistical | Z-score, robust z-score (MAD), IQR |
| Machine learning | Isolation Forest, LOF, autoencoders |

⚠️ **An anomaly is not automatically an error.** It might be a typo, fraud, a VIP customer, or a real event. **Investigate before you delete!**

In [ ]:
cols = ["quantity", "unit_price", "delivery_days", "session_minutes"]
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, c in zip(axes, cols):
    sns.boxplot(y=df[c], ax=ax)
    ax.set_title(c)
plt.tight_layout()
plt.show()

### 3.1 Z-score vs robust z-score

📖 Both ask the same question: **how far is this value from "normal"?** They differ in what they treat as "normal".

| | "Normal" (centre) | "Typical distance" (spread) | Usual rule |
|---|---|---|---|
| **Z-score** | mean | standard deviation | \|z\| > 3 → outlier |
| **Robust z-score** | median | MAD (median absolute deviation) | \|robust z\| > 3.5 → outlier |

⚠️ **The catch with the z-score:** the outliers themselves pull the **mean** up and blow up the **std**. The yardstick gets stretched by the very values you're trying to catch, so they can **hide themselves** (this is called *masking*). The median and MAD barely move when you add an outlier.

In [ ]:
def zscore(s):
    return (s - s.mean()) / s.std()

def robust_z(s):
    # median & MAD are not distorted by the outliers themselves
    med = s.median()
    mad = (s - med).abs().median()
    return 0.6745 * (s - med) / mad   # 0.6745 makes it comparable to a normal z-score

# Toy example: 10 deliveries, one is obviously wrong
toy = pd.Series([3, 4, 4, 5, 5, 5, 6, 6, 7, 60], name="delivery_days")
pd.DataFrame({"delivery_days": toy,
              "z": zscore(toy).round(1),
              "robust_z": robust_z(toy).round(1),
              "flag_z (>3)": zscore(toy).abs() > 3,
              "flag_robust (>3.5)": robust_z(toy).abs() > 3.5})

👀 The 60-day delivery is obviously wrong, but its **z-score is only ≈ 2.8, so the z-score misses it**. That one value pulled the mean from 5 up to 10.5 and inflated the std. The robust z-score (≈ 37) flags it without hesitation.

Now the real data. Let's turn each rule into a **cut-off in days**, so it's easy to read:

In [ ]:
s = df["delivery_days"]
mad = (s - s.median()).abs().median()
summary = pd.DataFrame({
    "centre":         [s.mean(), s.median()],
    "spread":         [s.std(), mad / 0.6745],
    "flags if above (days)": [s.mean() + 3 * s.std(), s.median() + 3.5 * mad / 0.6745],
    "orders flagged": [(zscore(s).abs() > 3).sum(), (robust_z(s).abs() > 3.5).sum()],
}, index=["Z-score (mean, std)", "Robust z (median, MAD)"]).round(1)
summary

💡 **How to read this:** the outliers inflate the std, so the z-score only raises the alarm above **~30 days**, while the robust rule draws the line at **~16 days**, which is much closer to what a human would call "late". Here both catch the same orders because the bad ones are extreme (60–200 days). With **subtler** problems (a few 20-day deliveries, say), only the robust rule would catch them.

👉 **Rule of thumb:** use the z-score on clean, roughly bell-shaped data, and the **robust z-score when outliers or skew may be present** (which is most real data).

### 3.2 IQR method (the boxplot rule)

In [ ]:
def iqr_outliers(s, k=1.5):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return (s < q1 - k * iqr) | (s > q3 + k * iqr)

print("IQR outliers in total_amount       :", iqr_outliers(df["total_amount"]).sum())
print("IQR outliers in log(total_amount)  :", iqr_outliers(np.log(df["total_amount"])).sum())

⚠️ On **skewed** data, IQR flags hundreds of perfectly normal (just big) orders. Transform first (log) or use **context**.

### 3.3 Contextual anomalies: compare like with like

In [ ]:
# Compare each order's price to the typical price of THE SAME product
df["price_vs_product"] = df["unit_price"] / df.groupby("product_id")["unit_price"].transform("median")
flag = df["price_vs_product"] > 2
x = np.arange(len(df))

fig, ax = plt.subplots(figsize=(11, 4))
ax.scatter(x[~flag], df.loc[~flag, "price_vs_product"], s=4, alpha=0.3, color="gray", label="normal price")
ax.scatter(x[flag], df.loc[flag, "price_vs_product"], s=45, color="red", label=f"> 2x the product's usual price ({flag.sum()} orders)")
ax.axhline(2, ls="--", color="red", lw=1)
ax.set_yscale("log")
ax.set_yticks([1, 2, 5, 10, 20, 50], labels=["1x", "2x", "5x", "10x", "20x", "50x"])
ax.set(xlabel="order", ylabel="price / product's median price (log scale)",
       title="Each order's price compared with the usual price of THE SAME product (1x = normal)")
ax.legend(loc="center right")
plt.show()

df.loc[df["price_vs_product"] > 2, ["order_id", "category", "product_id", "unit_price", "price_vs_product", "amount_mismatch"]].head()

### 3.4 Machine learning: Isolation Forest
📖 The idea is to split the data at random, again and again. **Anomalies are isolated in very few splits** because they sit far from everybody else. It also works across **many columns at once**.

In [ ]:
from sklearn.ensemble import IsolationForest

features = ["quantity", "unit_price", "discount", "shipping_cost", "delivery_days", "session_minutes", "total_amount"]
iso = IsolationForest(contamination=0.015, random_state=42)
df["iso_anomaly"] = iso.fit_predict(df[features]) == -1
df["iso_score"] = -iso.score_samples(df[features])     # higher = more anomalous

print("Flagged:", df["iso_anomaly"].sum())
df.sort_values("iso_score", ascending=False)[features + ["amount_mismatch", "iso_score"]].head(10)

Remember the `amount_mismatch` flag from Session 1? Let's see whether the model found those rows **without being told**:

In [ ]:
pd.crosstab(df["amount_mismatch"], df["iso_anomaly"], margins=True)

In [ ]:
flags = pd.DataFrame({
    "rule_delivery>30": df["delivery_days"] > 30,
    "rule_session>180": df["session_minutes"] > 180,
    "robust_z_quantity": robust_z(df["quantity"]).abs() > 3.5,
    "price_vs_product>2": df["price_vs_product"] > 2,
    "amount_mismatch": df["amount_mismatch"],
    "isolation_forest": df["iso_anomaly"],
})
df["n_flags"] = flags.sum(axis=1)
print(flags.sum().to_string(), end="\n\n")
print(df["n_flags"].value_counts().sort_index().to_string())

### 💡 What do we do with anomalies?
| Finding | Action |
|---|---|
| Data-entry error with a known fix | Fix it (e.g. discount 15 → 0.15) |
| Error, can't be fixed | Remove it, and **log** that you did |
| Real but extreme | Keep it, maybe **cap** (winsorize) or **log-transform** it for modeling |
| Suspicious (fraud?) | **Escalate** to the business |

### 🧪 Exercise 3.1: Build a review queue

Create `review_queue` with all orders flagged by **at least 2 methods**, sorted by `total_amount` (highest first). How many orders need review?

In [ ]:
# your code here

---
## 4. 🏗️ From notebook to pipeline: data-engineering basics *(⏱ 10 min)*

A notebook is great for exploring, but you **can't rerun 40 cells by hand every month**.

📖 **Principles**
- **ETL / ELT**: Extract → Transform → Load
- **Functions**, not copy-paste: each step is a function, chained with `.pipe()`
- **Idempotent**: running it twice gives the same result
- **Validate** the output automatically
- **Layers** (the *medallion architecture*):
  `🥉 Bronze (raw)` → `🥈 Silver (clean)` → `🥇 Gold (business-ready features/aggregates)`

In [ ]:
CITY_FIX = {"Ny": "New York", "Nyc": "New York", "La": "Los Angeles", "Seatle": "Seattle"}
CATEGORY_FIX = {"Electronic": "Electronics", "Home And Kitchen": "Home & Kitchen", "Home&Kitchen": "Home & Kitchen",
                "Clothes": "Clothing", "Book": "Books", "Sport": "Sports"}
BOOL_MAP = {"yes": True, "y": True, "1": True, "true": True, "no": False, "n": False, "0": False, "false": False}


def deduplicate(d):
    d = d.drop_duplicates()
    order = d.isna().sum(axis=1).sort_values(kind="stable").index
    return d.loc[order].drop_duplicates(subset="order_id").sort_index().drop(columns=["currency", "notes"])


def fix_types(d):
    return d.assign(
        unit_price=pd.to_numeric(d["unit_price"].astype(str).str.replace(r"[^0-9.]", "", regex=True)),
        customer_age=pd.to_numeric(d["customer_age"], errors="coerce"),
        order_date=pd.to_datetime(d["order_date"], format="mixed"),
        returned=d["returned"].astype(str).str.strip().str.lower().map(BOOL_MAP).astype(bool),
    )


def standardize(d):
    return d.assign(
        city=d["city"].str.strip().str.title().replace(CITY_FIX),
        category=d["category"].str.strip().str.title().replace(CATEGORY_FIX),
        payment_method=d["payment_method"].replace({"?": np.nan}),
    )


def apply_rules(d):
    d = d.copy()
    d.loc[~d["customer_age"].between(18, 100), "customer_age"] = np.nan
    d.loc[d["discount"] > 1, "discount"] /= 100
    d = d[d["quantity"] > 0]
    expected = d["quantity"] * d["unit_price"] * (1 - d["discount"]) + d["shipping_cost"]
    return d.assign(amount_mismatch=(expected - d["total_amount"]).abs() > 0.05)


def handle_missing(d):
    d = d.dropna(subset=["order_date"]).copy()
    d["customer_age"] = d["customer_age"].fillna(d.groupby("customer_id")["customer_age"].transform("median"))
    d["customer_age"] = d["customer_age"].fillna(d["customer_age"].median()).round()
    for col in ["delivery_days", "shipping_cost"]:
        d[col] = d[col].fillna(d.groupby("shipping_method")[col].transform("median"))
    d["payment_method"] = d["payment_method"].fillna("Unknown")
    d["has_rating"] = d["customer_rating"].notna()
    return d


def clean_orders(raw):
    return (raw.pipe(deduplicate).pipe(fix_types).pipe(standardize)
               .pipe(apply_rules).pipe(handle_missing).reset_index(drop=True))

In [ ]:
def validate_orders(d):
    checks = {
        "order_id is unique": d["order_id"].is_unique,
        "no missing except rating": d.drop(columns="customer_rating").notna().all().all(),
        "quantity > 0": (d["quantity"] > 0).all(),
        "discount in [0, 1]": d["discount"].between(0, 1).all(),
        "age in [18, 100]": d["customer_age"].between(18, 100).all(),
        "known cities only": d["city"].isin(["New York", "Los Angeles", "Chicago", "Houston", "Phoenix", "Seattle"]).all(),
        "dates in 2024": d["order_date"].dt.year.eq(2024).all(),
    }
    return pd.Series(checks, name="passed")


raw = pd.read_csv(DATA / "orders_raw.csv")      # 🥉 bronze
silver = clean_orders(raw)                       # 🥈 silver
print("Pipeline output:", silver.shape, "| Session 1 output:", df.shape[0], "rows")
validate_orders(silver)

### 🥇 Gold layer: feature engineering

In [ ]:
def add_features(d):
    return d.assign(
        order_month=d["order_date"].dt.month,
        weekday=d["order_date"].dt.day_name(),
        is_weekend=d["order_date"].dt.dayofweek >= 5,
        holiday_season=d["order_date"].dt.month.isin([11, 12]),
        late_delivery=d["delivery_days"] > 7,
        order_size=pd.cut(d["total_amount"], bins=[0, 50, 200, 1000, np.inf],
                          labels=["small", "medium", "large", "huge"]),
    )


gold_orders = silver.pipe(add_features)

# Customer-level table (RFM: Recency, Frequency, Monetary), a classic for segmentation
snapshot = gold_orders["order_date"].max() + pd.Timedelta(days=1)
gold_customers = gold_orders.groupby("customer_id").agg(
    recency_days=("order_date", lambda s: (snapshot - s.max()).days),
    frequency=("order_id", "count"),
    monetary=("total_amount", "sum"),
    avg_rating=("customer_rating", "mean"),
    return_rate=("returned", "mean"),
)
gold_orders.to_parquet(DATA / "gold_orders.parquet", index=False)
gold_customers.to_parquet(DATA / "gold_customers.parquet")
gold_customers.sort_values("monetary", ascending=False).head()

### 🧪 Exercise 4.1: Add a validation rule

Add a check **"delivery_days between 1 and 60"** to `validate_orders` and rerun it on `silver`. Does it pass? Why or why not?

In [ ]:
# your code here

### 📏 Feature scaling (normalization)
📖 Look at the customer table: `recency_days` is in **days** (1–365), `frequency` in **orders** (1–15),
`monetary` in **dollars** (up to tens of thousands). Many methods compare customers by **distance**
(k-means clustering, KNN, PCA, neural networks, regularized regression). Without scaling, the column with the **biggest numbers wins**,
just because of its units.

| Method | Formula | Result | Use when |
|---|---|---|---|
| **Min-max** | (x − min) / (max − min) | 0 to 1 | You need a fixed range, few outliers |
| **Standardization** (z-score) | (x − mean) / std | mean 0, std 1 | The default for most ML |
| **Robust scaling** | (x − median) / IQR | median 0 | Outliers are present |
| **Log transform** | log(1 + x) | Compresses the long tail | Right-skewed data (money, counts) |

In [ ]:
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler

rfm = gold_customers[["recency_days", "frequency", "monetary"]]
display(rfm.describe().loc[["min", "50%", "max", "std"]])

# How much of the total variance (the "spread" a distance-based model sees) does each column contribute?
(rfm.var() / rfm.var().sum()).map("{:.2%}".format)

🤔 `monetary` contributes almost **all** of the variance. A clustering model would basically ignore recency and frequency.

In [ ]:
scalers = {"Min-max": MinMaxScaler(), "Standard": StandardScaler(), "Robust": RobustScaler()}
scaled = {name: pd.DataFrame(s.fit_transform(rfm), columns=rfm.columns, index=rfm.index)
          for name, s in scalers.items()}

pd.concat({name: d.agg(["min", "median", "max"]) for name, d in scaled.items()}).round(2)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
sns.histplot(rfm["monetary"], bins=50, ax=axes[0]).set_title("Raw ($)")
sns.histplot(scaled["Min-max"]["monetary"], bins=50, ax=axes[1]).set_title("Min-max (0–1)")
sns.histplot(scaled["Standard"]["monetary"], bins=50, ax=axes[2]).set_title("Standardized (z)")
sns.histplot(np.log1p(rfm["monetary"]), bins=50, ax=axes[3]).set_title("log(1 + x)")
plt.tight_layout()
plt.show()

print("Variance share after standardizing:")
print((scaled["Standard"].var() / scaled["Standard"].var().sum()).map("{:.1%}".format).to_string())

⚠️ **Three things to remember**
1. Min-max and standardization change the **units, not the shape**: skewed data stays skewed (see the first three charts). Only a **log** transform changes the shape.
2. **Tree-based models** (decision trees, random forests, gradient boosting) don't need scaling.
3. **Avoid data leakage:** in machine learning, `fit` the scaler on the **training data only**, then `transform` the test data with it.

### 🧪 Exercise 4.2: Prepare RFM for clustering

Create `rfm_ready`: first apply `np.log1p` to `frequency` and `monetary` (they are right-skewed),
then standardize all three columns with `StandardScaler`. Check that every column has mean ≈ 0 and std ≈ 1.

In [ ]:
# your code here

---
## 5. 🎤 Wrap-up: insights for the CEO *(⏱ 5 min)*

A good insight is **specific, quantified and actionable**. Use the data to complete the three below:
1. 📦 *Delivery speed:* every extra day of delivery costs us … rating points, and low ratings come with … × more returns.
2. 🎄 *Seasonality:* Nov–Dec generate …% of the yearly revenue.
3. 🚨 *Data risk:* … orders need manual review for possible fraud or data-entry errors.

### 🧪 Exercise 5.1: Back your insights with numbers

Compute the numbers for the three insights above.

In [ ]:
# your code here

---
## ✅ Recap
| Step | Key ideas |
|---|---|
| **EDA** | Univariate → bivariate → multivariate; median for skewed data; groupby & pivot |
| **Correlation** | Pearson (linear) vs Spearman (monotonic, robust); r≈0 ≠ no relation; outliers; **correlation ≠ causation** |
| **Anomalies** | Rules → statistics (robust z, IQR) → context → ML (Isolation Forest); investigate, don't auto-delete |
| **Engineering** | Functions + `.pipe()`, validation, bronze/silver/gold, feature engineering, scaling (min-max, standard, robust, log) |

### 🚀 Where to go next
- **Machine learning**: predict returns or ratings with `scikit-learn`
- **Data validation libraries**: `pandera`, Great Expectations
- **Bigger data**: Polars, DuckDB, Spark
- **Communicating results**: Power BI, Streamlit dashboards
- **SQL**: the language of data engineering

🙏 **Thank you!**